##### Importação dos Pacotes

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import monotonically_increasing_id, current_timestamp, date_format
import geopandas as gpd

##### Catalog

In [0]:
path_bronze_ibge_shp = "/Volumes/01_bronze/schemas/ibge/shp_geografico"
path_bronze_ibge_dtb = "/Volumes/01_bronze/schemas/ibge/dtb_territorio"
path_silver_ibge_dtb = "/Volumes/02_silver/schemas/ibge/dtb_territorio"
path_silver_ibge_geo = "/Volumes/02_silver/schemas/ibge/shp_geografico"
path_gold_dimensao   = '/Volumes/03_gold/schemas/dimensao'

##### IBGE - GEO: consolida DataFrame todos arquivos Parquet

In [0]:
# DataFrame dos arquivos Parquet na camada: Silver
geo_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(path_silver_ibge_geo)
)

In [0]:
## Validação das colunas
#print(geo_df.columns)

##### IBGE_GEO: reprojeção Cartográfica para Metros (EPSG:4674) / Cálcula Centróide

In [0]:
# Converte para Pandas
geo = geo_df.toPandas()

# Calcula geometria do polígono (para SIRGAS 2000)
gdf = gpd.GeoDataFrame(
    geo,
    geometry = gpd.GeoSeries.from_wkt(geo["geometry"]),
    crs="EPSG:4674"
)

# Projeção Cartográfica para abranger todo Brasil (EPSG:31983 - focado em São Paulo)
gdf = gdf.to_crs("EPSG:5880")

# Cria coluna do Centroíde
gdf["Centroide"] = gdf.geometry.centroid

# Extrai coordenadas métricas
#gdf["Centroide_X"] = gdf["Centroide"].x
#gdf["Centroide_Y"] = gdf["Centroide"].y

In [0]:
## Validação da Projeção Cartográfico do DataFrame
#gdf.crs

In [0]:
# Converte geometria e centróide para WKT
# *necessário para transformar informações geográfica em Strings*
gdf["Geometria_wkt"] = gdf.geometry.to_wkt()
gdf["Centroide_wkt"] = gdf["Centroide"].to_wkt()

In [0]:
# Exclui colunas do DataFrame
gdf_final = gdf.drop(columns=["CD_CONCU","NM_CONCU","geometry", "Centroide"])

In [0]:
# Transforma em Spark
spark_df = spark.createDataFrame(gdf_final)
# Para transformar em tabela temporária (para utilizar no SQL)
spark_df.createOrReplaceTempView("df_shp")

##### IBGE_DTB: carrega tabela de Territórios (DTB)

In [0]:
# Consolida arquivos na camada Silver (fato) em um único Dataframe
df_dtb = spark.read.format("parquet").load(path_silver_ibge_dtb)

In [0]:
# Transformar em tabela temporária (para utilizar no SQL)
df_dtb.createOrReplaceTempView("df_dtb")

##### Join entre as duas tabelas (IBGE: GEO & DTB)

In [0]:
geo = spark.sql("""
                SELECT
                   S.*
                  ,D.COD_TSE
                  --,D.MUNICIPIO
                  FROM df_shp S
                  LEFT JOIN df_dtb D ON S.CD_MUN = D.COD_MUN 
              """)
# Transformar em tabela temporária (para utilizar no SQL)

##### Transformação/Padronização

In [0]:
# Todos os valores "Sem Info" são convertidos para "None"
geo = geo.replace("Sem Info", None)

In [0]:
# Cria coluna Id (incremental)
dim_geo = geo.withColumn("IdGeografia", monotonically_increasing_id()+1) \
             .withColumn("DtAtualizacao",
                            date_format(current_timestamp(), "dd/MM/yyyy HH:mm:ss"))

In [0]:
# Validação do formato dos campos
#dim_geo.printSchema()

In [0]:
# Conversão dos campos do tipo "String" para "Inteiro"
cols_int = [
    "CD_UF",
    "CD_RGI",
    "CD_RGINT",
    "CD_MUN",
    "COD_TSE"
]

dim_geo = dim_geo.select(
    *[
        col(c).cast("int").alias(c) if c in cols_int else col(c)
        for c in dim_geo.columns
    ]
)

In [0]:
# Reordena e Renomeia as colunas
dim_geo = dim_geo.select(
            col("IdGeografia").alias("IdGeografia"),
            col("SIGLA_RG").alias("SiglaRegiao"),
            col("CD_UF").alias("SkUF"),
            col("NM_UF").alias("UF"),
            col("SIGLA_UF").alias("SiglaUF"),
            col("CD_RGI").alias("SkRegiaoImediata"),
            col("NM_RGI").alias("RegiaoImediata"),
            col("CD_RGINT").alias("SkRegiaoIntermediaria"),
            col("NM_RGINT").alias("RegiaoIntermediaria"),
            col("CD_MUN").alias("SkMunicipio"),
            col("COD_TSE").alias("SkMunicipioTSE"),
            col("NM_MUN").alias("Municipio"),
            col("AREA_KM2").alias("AreaKm"),
            col("Geometria_wkt").alias("Geometria"),
            col("Centroide_wkt").alias("Centroide"),
            col("DtAtualizacao").alias("DtAtualizacao"),
)

In [0]:
# Validação do DataFrame
#display(dim_geo.filter(dim_geo['SiglaUF'] == 'TO'))

In [0]:
# Salva na camada: Gold (Dimensão)
dim_geo.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("SiglaUF") \
        .save(f"{path_gold_dimensao}/dim_geografia")

##### Fim